# 06 — Time Series Basics

## What is a time series?

A **time series** is a sequence of values indexed by time. Unlike tabular data
where rows are independent, time series have **temporal dependencies** —
yesterday's close price influences today's.

## Key components

| Component | Description | Example |
|-----------|-------------|----------|
| **Trend** | Long-term upward/downward direction | Stock rising over years |
| **Seasonality** | Regular periodic pattern | Retail sales peak in December |
| **Lag** | Correlation with a past value at step $k$ | Today correlated with yesterday |
| **Noise** | Random fluctuation | Day-to-day random movement |

## Stationarity

A time series is **stationary** if its statistical properties (mean, variance)
do not change over time. Most real-world series are **non-stationary** —
they have changing means or variances, which breaks the assumptions of many
forecasting models.

→ This is the core problem that **RevIN** is designed to solve.

---

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from utils.ml_utils import create_sequences
from utils.plotting import plot_line, plot_rolling_stats

In [ ]:
df            = pd.read_csv('../data/stock_sample.csv')
df['date']    = pd.to_datetime(df['date'])
df            = df.set_index('date')
df.head()

## Step 1 — Visualise the price series

In [ ]:
plot_line(
    df.index, df['close'],
    title='Stock Close Price over Time',
    xlabel='Date', ylabel='Price ($)',
    label='Close'
)

## Step 2 — Rolling mean (trend smoother)

The rolling mean averages the last `window` values at each time step.
It smooths out noise and reveals the underlying trend.

In [ ]:
plot_rolling_stats(df['close'].values, window=10, title='Close Price — Rolling Stats', ylabel='Price ($)')

## Step 3 — Lags and autocorrelation

A **lag-k autocorrelation** measures how correlated the series is with its
own value $k$ steps ago. High autocorrelation at lag 1 means yesterday's
price strongly predicts today's — which is typical for stock prices.

In [ ]:
close  = df['close'].values
lags   = range(1, 11)
corrs  = [np.corrcoef(close[k:], close[:-k])[0, 1] for k in lags]

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(lags, corrs, color='#4C72B0', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Autocorrelation by Lag', fontsize=13)
ax.set_xlabel('Lag (days)')
ax.set_ylabel('Correlation')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

for k, c in zip(lags, corrs):
    print(f'Lag {k:2d}: {c:.3f}')

## Step 4 — Creating sequences for forecasting

To train a supervised forecasting model, we convert the raw series into
(past window, future value) pairs using a **sliding window**.

In [ ]:
LOOKBACK = 10
HORIZON  = 1

X, y = create_sequences(close, lookback=LOOKBACK, horizon=HORIZON)

print(f'Series length  : {len(close)}')
print(f'X shape        : {X.shape}   (samples, lookback)')
print(f'y shape        : {y.shape}   (samples, horizon)')
print()
print('First window  X[0]:', X[0])
print('First target  y[0]:', y[0])

## Step 5 — Visualise one forecast window

In [ ]:
idx    = 50  # pick a window
window = X[idx]
target = y[idx][0]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(range(LOOKBACK), window, 'o-', color='#4C72B0', label='Input window')
ax.plot(LOOKBACK, target, 'D', color='red', markersize=10, label=f'Target (t+1)')
ax.axvline(LOOKBACK - 0.5, color='gray', linestyle='--', linewidth=1)
ax.set_title(f'Forecast window (sample {idx})', fontsize=13)
ax.set_xlabel('Time step')
ax.set_ylabel('Price ($)')
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

---
## Exercises

1. What is the lookback window size with the highest lag-1 autocorrelation for `volume`?
2. Try `HORIZON = 5` in `create_sequences`. What changes in the shape of `y`?
3. Plot the rolling mean with windows of 5, 15, and 30 days on the same chart.
   What does a longer window do to the smoothness?

---
**Next →** `07_distribution_shift.ipynb` — the key problem RevIN solves.